# M3: set-aware conditional low-rank Gaussian copula

M3 preserves M2's low-rank covariance law but first contextualizes each entity using every member of the complete physical group:

$$H_{g,\tau}^{(i)}=\operatorname{TransformerEncoder}_\theta(V_{g,\tau}^{(i)}),\qquad (\lambda_k,b_k)=h_\theta(H_k).$$

There are no positional encodings and no learned entity identifiers. Therefore, for a permutation matrix $P$, $R_\theta(PV)=P R_\theta(V)P^\mathsf T$. This is an equivariance property, not permission to evaluate smaller entity sets.

In [ ]:
from pathlib import Path
import sys, numpy as np
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'src')) if str(PROJECT_ROOT / 'src') not in sys.path else None
from simcast.config import SimcastConfig, deep_merge, load_config
from simcast.fm.cache import load_pit_library
from simcast.cli.train_dependence import train_from_config
from simcast.cli.evaluate import evaluate_from_config

CONFIG_FILE = 'configs/powertech2027/transformer.yaml'
OVERRIDES = ()
TRAIN_IF_MISSING = False
RUN_EVALUATION = False
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'notebook_walkthrough' / 'm3_set_aware_low_rank'
base = load_config(PROJECT_ROOT / CONFIG_FILE, overrides=OVERRIDES)
config = SimcastConfig.model_validate(deep_merge(base.model_dump(mode='python'), {'dependence': {'method': 'set_aware_low_rank'}}))
CACHE_DIR = PROJECT_ROOT / config.output.cache_dir / (config.output.cache_name or f'liander2024_{config.data.entity_type}')
library = load_pit_library(CACHE_DIR, access='training'); ds = library.dataset
entity_ids = [str(x) for x in ds.entity_id.values]; K_g = len(entity_ids)
assert entity_ids == config.protocol.ordered_entity_ids and K_g == config.protocol.entity_count
print(f'group={config.data.entity_type}, K_g={K_g}, heads={config.dependence.set_aware_low_rank.num_heads}')

## What M3 adds over M2

M2 can only compare entity-specific factor parameters after they have been produced independently. M3 allows the factor parameters for one entity to depend on contemporaneous frozen features of every other entity. Its scientific question is therefore narrower than “does attention work?”: does full-group forecast context improve the conditional representation of same-lead residual dependence beyond M1 and M2?

The complete-vector rule is essential here. Each attention operation receives $K_g$ entities; a case with a missing member is invalid rather than padded or shrunk in the study.

In [ ]:
z = np.asarray(ds['pit_z'].values)
valid = np.isfinite(z).all(axis=1)
print('cached tensor dimensions:', dict(ds.sizes))
print('full-group valid vectors:', int(valid.sum()), 'of', valid.size)
print('ordered entities:')
for index, entity in enumerate(entity_ids, start=1): print(f'{index:2d}. {entity}')
assert np.all(np.isfinite(z[valid]))

## Fitting and evaluation

As for M2, the objective is the Gaussian-copula pseudo-NLL and checkpoint selection uses only chronological validation cases. The evaluation then produces correlated uniforms using $R_{g,\tau}^{(i)}$, projects them through unchanged finite quantile grids, and scores the full spatial aggregate and joint entity vector.

In [ ]:
run_dir = OUTPUT_DIR / 'set_aware_low_rank'
if TRAIN_IF_MISSING and not run_dir.exists():
    run_dir = train_from_config(config, cache_dir=CACHE_DIR, output_dir=run_dir)
if RUN_EVALUATION:
    if not run_dir.exists(): raise FileNotFoundError('Set TRAIN_IF_MISSING=True or choose an existing M3 run.')
    evaluate_from_config(config, methods=('set_aware_low_rank',), method_runs={'set_aware_low_rank': run_dir}, cache_dir=CACHE_DIR, output_dir=OUTPUT_DIR / 'evaluation')
else:
    print('Read-only mode: no M3 training or evaluation artifact is written.')

In [ ]:
import pandas as pd
from IPython.display import display
metrics_file = OUTPUT_DIR / 'evaluation' / 'metrics_by_lead.csv'
if metrics_file.is_file():
    metrics = pd.read_csv(metrics_file)
    display(metrics.groupby('method', as_index=False).mean(numeric_only=True))
    metrics.pivot(index='lead', columns='method', values='mean_pinball').plot(title='M3 aggregate pinball loss by lead')
else:
    print('No notebook evaluation table yet. Read-only inspection does not create one.')